In [ ]:
# In[1]:

import pandas as pd
from pathlib import Path

# file list
files = {
    "metric.csv": {"name_col": "kpi_name"},
    "trace.csv": {"name_col": "trace_name"},
    "log.csv": {"name_col": "log_name"},
    "error_logs.csv": {"name_col": "message"},
}

# Load dataframes, parse timestamps, compute summaries
summaries = []
for fname, meta in files.items():
    path = Path(fname)
    if not path.exists():
        # If a file is missing, record a compact missing entry
        summaries.append({
            "filename": fname,
            "rows": 0,
            "unique_cmdb_ids": [],
            "unique_names": [],
            "min_timestamp_utc": pd.NaT,
            "max_timestamp_utc": pd.NaT,
        })
        continue

    df = pd.read_csv(path)
    # parse timestamps to UTC datetimes (Unix seconds)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        min_ts = df["timestamp"].min()
        max_ts = df["timestamp"].max()
    else:
        min_ts = pd.NaT
        max_ts = pd.NaT

    # unique cmdb_id (limit to first 20)
    if "cmdb_id" in df.columns:
        uniq_cmdb = list(pd.Index(df["cmdb_id"].dropna().unique())[:20])
    else:
        uniq_cmdb = []

    # unique KPI/trace/log name field values (limit to first 20)
    name_col = meta.get("name_col")
    if name_col and name_col in df.columns:
        uniq_names = list(pd.Index(df[name_col].dropna().unique())[:20])
    else:
        uniq_names = []

    summaries.append({
        "filename": fname,
        "rows": int(len(df)),
        "unique_cmdb_ids": uniq_cmdb,
        "unique_names": uniq_names,
        "min_timestamp_utc": min_ts,
        "max_timestamp_utc": max_ts,
    })

# Create a compact summary DataFrame (one row per file)
summary_df = pd.DataFrame(summaries)[[
    "filename", "rows", "unique_cmdb_ids", "unique_names", "min_timestamp_utc", "max_timestamp_utc"
]]

# Display the compact summary
summary_df

```
Out[1]:
```
```python
# Build a plain-English summary from the previously computed summary_df
def fmt_list(lst, max_items=6):
    if not lst:
        return "none"
    shown = lst[:max_items]
    more = len(lst) - len(shown)
    return ", ".join(shown) + (f", ... (+{more} more)" if more > 0 else "")

lines = []
for _, row in summary_df.iterrows():
    fname = row['filename']
    rows = int(row['rows'])
    cmdbs = fmt_list(row['unique_cmdb_ids'])
    names = fmt_list(row['unique_names'])
    min_ts = row['min_timestamp_utc']
    max_ts = row['max_timestamp_utc']
    if pd.isna(min_ts) or pd.isna(max_ts):
        time_range = "no timestamps"
    else:
        time_range = f"{pd.to_datetime(min_ts).isoformat()} to {pd.to_datetime(max_ts).isoformat()}"
    lines.append(f"{fname}: {rows} rows; cmdb_ids: {cmdbs}; names: {names}; time range (UTC): {time_range}.")

summary = "Summary of telemetry files (plain English):\n" + "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

filename  rows                                    unique_cmdb_ids                                       unique_names         min_timestamp_utc         max_timestamp_utc
0      metric.csv  1800  [adservice, cartservice, checkoutservice, curr...  [cpu, diskio, latency-50, latency-90, mem, soc... 2024-01-15 21:24:00+00:00 2024-01-15 21:48:00+00:00
1       trace.csv  3800  [checkoutservice, currencyservice, emailservic...  [trace.from_checkoutservice.duration_mean, tra... 2024-01-15 21:24:00+00:00 2024-01-15 21:48:00+00:00
2         log.csv   458  [adservice, cartservice, checkoutservice, curr...                   [log.error_count, log.row_count] 2024-01-15 21:24:00+00:00 2024-01-15 21:48:00+00:00
3  error_logs.csv     0                                                 []                                                 []                       NaT                       NaT```
```

In [ ]:
# In[2]:

import pandas as pd

def process_file(filename, name_col):
    df = pd.read_csv(filename)
    # 1) parse timestamp as UTC datetimes
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

    group_cols = ["cmdb_id", name_col]

    # 2) compute global P95 on the full series (per group)
    global_p95 = (
        df.groupby(group_cols)["value"]
        .quantile(0.95)
        .reset_index()
        .rename(columns={"value": "global_p95"})
    )

    # 3) filter to incident window (inclusive)
    start = pd.to_datetime("2024-01-15 21:21:06", utc=True)
    end = pd.to_datetime("2024-01-15 21:51:06", utc=True)
    df_window = df[(df["timestamp"] >= start) & (df["timestamp"] <= end)].copy()

    # 4) For windowed data compute aggregates relative to global_p95
    if not df_window.empty:
        # merge global p95 into window rows
        df_window = df_window.merge(global_p95, on=["cmdb_id", name_col], how="left")
        df_window["above_p95"] = df_window["value"] > df_window["global_p95"]
    else:
        # create empty with expected columns to allow groupby results (no rows)
        df_window = df_window.assign(global_p95=pd.Series(dtype=float), above_p95=pd.Series(dtype=bool))

    # Aggregates in window
    total_points = (
        df_window.groupby(["cmdb_id", name_col])["value"]
        .count()
        .rename("total_points_in_window")
        .reset_index()
    )
    points_above = (
        df_window.groupby(["cmdb_id", name_col])["above_p95"]
        .sum()
        .rename("points_above_p95")
        .reset_index()
    )
    earliest_anom = (
        df_window[df_window["above_p95"]]
        .groupby(["cmdb_id", name_col])["timestamp"]
        .min()
        .rename("earliest_anomaly_timestamp_utc")
        .reset_index()
    )
    max_value = (
        df_window.groupby(["cmdb_id", name_col])["value"]
        .max()
        .rename("max_value_in_window")
        .reset_index()
    )

    # Start from global_p95 (all groups seen in full series) then left-join window aggregates
    merged = global_p95.merge(total_points, on=["cmdb_id", name_col], how="left").merge(
        points_above, on=["cmdb_id", name_col], how="left"
    ).merge(earliest_anom, on=["cmdb_id", name_col], how="left").merge(
        max_value, on=["cmdb_id", name_col], how="left"
    )

    # Fill NaNs for counts with 0, keep timestamps as NaT if missing
    merged["total_points_in_window"] = merged["total_points_in_window"].fillna(0).astype(int)
    merged["points_above_p95"] = merged["points_above_p95"].fillna(0).astype(int)

    # Rename columns to requested output names
    merged = merged.rename(columns={"cmdb_id": "component", name_col: "name"})

    # Reorder columns
    merged = merged[
        [
            "component",
            "name",
            "global_p95",
            "total_points_in_window",
            "points_above_p95",
            "earliest_anomaly_timestamp_utc",
            "max_value_in_window",
        ]
    ]

    # 5) sort and limit to top 20
    merged_sorted = merged.sort_values(
        by=["points_above_p95", "max_value_in_window"], ascending=[False, False]
    ).head(20).reset_index(drop=True)

    # 6) counts
    num_groups = int(len(global_p95))
    groups_with_anom = int((merged["points_above_p95"] > 0).sum())

    counts = pd.Series(
        {
            "num_groups_scanned": num_groups,
            "groups_with_points_above_p95": groups_with_anom,
        }
    )

    return merged_sorted, counts

# Process files
metric_summary_top, metric_counts = process_file("metric.csv", "kpi_name")
trace_summary_top, trace_counts = process_file("trace.csv", "trace_name")
log_summary_top, log_counts = process_file("log.csv", "log_name")

# Return compact results (variables will display in the IPython environment)
metric_summary_top, metric_counts, trace_summary_top, trace_counts, log_summary_top, log_counts

```
Out[2]:
```
summary = (
    "Summary of anomaly scan (incident window 2024-01-15 21:21:06–21:51:06 UTC):\n\n"
    "metric.csv:\n"
    "- Scanned 72 (component,kpi) groups; 60 groups had points above their global P95 during the window.\n"
    "- Top anomalous KPI is memory (mem) across many services (examples: adservice, recommendationservice, "
    "cartservice, emailservice, currencyservice, paymentservice, frontend, checkoutservice, productcatalogservice, shippingservice, redis). "
    "These groups show 25 points in-window and typically 2 points above their global P95; earliest memory anomalies appear ~21:28–21:47 UTC.\n"
    "- Other smaller anomalies appear for diskio, workload and CPU in a few services.\n\n"
    "trace.csv:\n"
    "- Scanned 152 (component,trace) groups; 104 groups had points above global P95.\n"
    "- Top anomalies are increased trace row_count values centered on frontend and downstream services (frontendservice, productcatalogservice, currencyservice, "
    "recommendationservice, checkoutservice, etc.). These rows also show 25 points in-window with ~2 points above P95; earliest trace row_count spikes ~21:25–21:36 UTC.\n\n"
    "log.csv:\n"
    "- Scanned 20 (component,log) groups; 9 groups had points above global P95.\n"
    "- The main signal is log.row_count increases in frontend, currencyservice, cartservice, recommendationservice, shippingservice, adservice, checkoutservice, paymentservice, emailservice (mostly 25 points in-window and ~2 above P95).\n"
    "- log.error_count remained effectively zero (no error-count spikes).\n\n"
    "Overall interpretation:\n"
    "- Telemetry shows a correlated surge in request/row_count (traces and logs) starting around ~21:25–21:36 UTC, followed by widespread memory increases across many services during the incident window.\n"
    "- This pattern points to elevated workload/traffic causing resource pressure (memory and some diskio/cpu), rather than localized socket or error-log failures.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                component      name    global_p95  total_points_in_window  points_above_p95 earliest_anomaly_timestamp_utc  max_value_in_window
0               adservice       mem  1.117398e+08                      25                 2      2024-01-15 21:38:00+00:00         1.117921e+08
1   recommendationservice       mem  4.509696e+07                      25                 2      2024-01-15 21:47:00+00:00         4.511744e+07
2             cartservice       mem  4.359283e+07                      25                 2      2024-01-15 21:44:00+00:00         4.362336e+07
3            emailservice       mem  4.325200e+07                      25                 2      2024-01-15 21:43:00+00:00         4.326605e+07
4         currencyservice       mem  4.097597e+07                      25                 2      2024-01-15 21:37:00+00:00         4.111551e+07
5          paymentservice       mem  3.876485e+07                      25                 2      2024-01-15 21:44:00+00:00         3.877984e+07
6                frontend       mem  1.659449e+07                      25                 2      2024-01-15 21:28:00+00:00         1.669789e+07
7         checkoutservice       mem  1.651220e+07                      25                 2      2024-01-15 21:46:00+00:00         1.655050e+07
8   productcatalogservice       mem  1.042969e+07                      25                 2      2024-01-15 21:29:00+00:00         1.043845e+07
9         shippingservice       mem  8.991867e+06                      25                 2      2024-01-15 21:26:00+00:00         9.182822e+06
10                  redis       mem  3.080629e+06                      25                 2      2024-01-15 21:41:00+00:00         3.106543e+06
11              adservice    diskio  7.145027e+04                      25                 2      2024-01-15 21:41:00+00:00         7.592993e+04
12                  redis    diskio  2.002572e+04                      25                 2      2024-01-15 21:39:00+00:00         2.093025e+04
13  productcatalogservice  workload  1.435848e+02                      25                 2      2024-01-15 21:38:00+00:00         1.439564e+02
14        currencyservice  workload  7.868004e+01                      25                 2      2024-01-15 21:24:00+00:00         8.206359e+01
15            cartservice  workload  2.586088e+01                      25                 2      2024-01-15 21:38:00+00:00         2.640000e+01
16        checkoutservice       cpu  1.978319e+01                      25                 2      2024-01-15 21:42:00+00:00         1.999501e+01
17        currencyservice       cpu  1.881616e+01                      25                 2      2024-01-15 21:24:00+00:00         1.906427e+01
18  recommendationservice  workload  1.873964e+01                      25                 2      2024-01-15 21:32:00+00:00         1.905110e+01
19              adservice  workload  1.509255e+01                      25                 2      2024-01-15 21:38:00+00:00         1.515448e+01, num_groups_scanned              72
groups_with_points_above_p95    60
dtype: int64,                 component                                        name  global_p95  total_points_in_window  points_above_p95 earliest_anomaly_timestamp_utc  max_value_in_window
0         frontendservice        trace.from_frontendservice.row_count      7800.0                      25                 2      2024-01-15 21:26:00+00:00               7822.0
1         frontendservice          trace.to_frontendservice.row_count      7800.0                      25                 2      2024-01-15 21:26:00+00:00               7822.0
2         frontendservice    trace.to_productcatalogservice.row_count      3675.0                      25                 2      2024-01-15 21:25:00+00:00               3699.0
3   productcatalogservice        trace.from_frontendservice.row_count      3675.0                      25                 2      2024-01-15 21:25:00+00:00               3699.0
4         currencyservice        trace.from_frontendservice.row_count      2296.4                      25                 2      2024-01-15 21:33:00+00:00               2337.0
5         frontendservice          trace.to_currencyservice.row_count      2296.4                      25                 2      2024-01-15 21:33:00+00:00               2337.0
6         frontendservice                   trace.from_root.row_count       811.6                      25                 2      2024-01-15 21:36:00+00:00                833.0
7                    root          trace.to_frontendservice.row_count       811.6                      25                 2      2024-01-15 21:36:00+00:00                833.0
8         frontendservice    trace.to_recommendationservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
9   productcatalogservice  trace.from_recommendationservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
10  recommendationservice        trace.from_frontendservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
11  recommendationservice  trace.from_recommendationservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
12  recommendationservice    trace.to_productcatalogservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
13  recommendationservice    trace.to_recommendationservice.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                564.0
14        checkoutservice        trace.from_checkoutservice.row_count       304.4                      25                 2      2024-01-15 21:36:00+00:00                353.0
15        checkoutservice          trace.to_checkoutservice.row_count       304.4                      25                 2      2024-01-15 21:36:00+00:00                353.0
16        checkoutservice                   trace.from_root.row_count       147.0                      25                 2      2024-01-15 21:30:00+00:00                168.0
17                   root          trace.to_checkoutservice.row_count       147.0                      25                 2      2024-01-15 21:30:00+00:00                168.0
18        checkoutservice          trace.to_currencyservice.row_count       122.2                      25                 2      2024-01-15 21:30:00+00:00                145.0
19        currencyservice        trace.from_checkoutservice.row_count       122.2                      25                 2      2024-01-15 21:30:00+00:00                145.0, num_groups_scanned              152
groups_with_points_above_p95    104
dtype: int64,                 component             name  global_p95  total_points_in_window  points_above_p95 earliest_anomaly_timestamp_utc  max_value_in_window
0                frontend    log.row_count      2683.4                      25                 2      2024-01-15 21:36:00+00:00                 2739
1         currencyservice    log.row_count      2383.8                      25                 2      2024-01-15 21:26:00+00:00                 2428
2             cartservice    log.row_count       773.8                      25                 2      2024-01-15 21:36:00+00:00                  786
3   recommendationservice    log.row_count       562.0                      25                 2      2024-01-15 21:25:00+00:00                  564
4         shippingservice    log.row_count       474.0                      25                 2      2024-01-15 21:36:00+00:00                  486
5               adservice    log.row_count       452.8                      25                 2      2024-01-15 21:29:00+00:00                  465
6         checkoutservice    log.row_count       112.2                      25                 2      2024-01-15 21:36:00+00:00                  125
7          paymentservice    log.row_count        74.8                      25                 2      2024-01-15 21:36:00+00:00                   84
8            emailservice    log.row_count        37.4                      25                 2      2024-01-15 21:36:00+00:00                   42
9                   redis    log.row_count         5.0                       4                 0                            NaT                    5
10              adservice  log.error_count         0.0                      25                 0                            NaT                    0
11            cartservice  log.error_count         0.0                      25                 0                            NaT                    0
12        checkoutservice  log.error_count         0.0                      25                 0                            NaT                    0
13        currencyservice  log.error_count         0.0                      25                 0                            NaT                    0
14           emailservice  log.error_count         0.0                      25                 0                            NaT                    0
15               frontend  log.error_count         0.0                      25                 0                            NaT                    0
16         paymentservice  log.error_count         0.0                      25                 0                            NaT                    0
17  recommendationservice  log.error_count         0.0                      25                 0                            NaT                    0
18                  redis  log.error_count         0.0                       4                 0                            NaT                    0
19        shippingservice  log.error_count         0.0                      25                 0                            NaT                    0, num_groups_scanned              20
groups_with_points_above_p95     9
dtype: int64)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# 1) Read metric.csv and parse timestamp as UTC
df = pd.read_csv("metric.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# 2) Compute global P95 per (cmdb_id, kpi_name) using the full series
p95_full = (
    df.groupby(["cmdb_id", "kpi_name"])["value"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"value": "global_p95"})
)

# Restrict to groups where cmdb_id is one of the five candidates (these are the groups to report)
candidate_groups = p95_full[p95_full["cmdb_id"].isin(candidates)].reset_index(drop=True)

# 3) Filter metric.csv to the incident window (inclusive)
start = pd.to_datetime("2024-01-15 21:21:06", utc=True)
end = pd.to_datetime("2024-01-15 21:51:06", utc=True)
df_window = df[(df["timestamp"] >= start) & (df["timestamp"] <= end)].copy()

# 4-5) For each candidate (cmdb_id, kpi_name) compute anomalies and runs
records = []
for _, grp in candidate_groups.iterrows():
    comp = grp["cmdb_id"]
    kpi = grp["kpi_name"]
    global_p95 = grp["global_p95"]

    mask = (df_window["cmdb_id"] == comp) & (df_window["kpi_name"] == kpi)
    sub = df_window[mask].sort_values("timestamp")

    total_points = int(sub.shape[0])
    # points above global P95
    anomalies = sub[sub["value"] > global_p95].copy()
    total_anomalous = int(anomalies.shape[0])

    # max value in window (NaN if no points)
    max_value = sub["value"].max() if total_points > 0 else np.nan

    # compute runs of consecutive anomalous timestamps (consecutive = 1 minute apart)
    if total_anomalous == 0:
        num_runs = 0
        longest_run = 0
        earliest_run_start = pd.NaT
    else:
        anomalies = anomalies.sort_values("timestamp").reset_index(drop=True)
        diffs = anomalies["timestamp"].diff()
        # start a new run when diff is not exactly 1 minute (or is NaT for first row)
        new_run = (diffs != pd.Timedelta(minutes=1)) | (diffs.isna())
        run_id = new_run.cumsum()
        run_sizes = anomalies.groupby(run_id).size().tolist()
        num_runs = len(run_sizes)
        longest_run = int(max(run_sizes)) if run_sizes else 0
        # earliest run start = first timestamp of the first run (since sorted)
        earliest_run_start = anomalies.loc[0, "timestamp"]

    # max exceed ratio
    if pd.notna(max_value) and global_p95 != 0:
        max_exceed_ratio = (float(max_value) - float(global_p95)) / float(global_p95)
    else:
        max_exceed_ratio = np.nan

    records.append(
        {
            "component": comp,
            "kpi_name": kpi,
            "global_p95": float(global_p95),
            "total_points_in_window": total_points,
            "total_anomalous_points": total_anomalous,
            "number_of_anomalous_runs": num_runs,
            "longest_run_length": longest_run,
            "earliest_run_start_utc": earliest_run_start,
            "max_value_in_window": float(max_value) if pd.notna(max_value) else np.nan,
            "max_exceed_ratio": float(max_exceed_ratio) if pd.notna(max_exceed_ratio) else np.nan,
        }
    )

# Build result DataFrame
result_df = pd.DataFrame.from_records(records)

# 6) Sort by total_anomalous_points desc, then max_exceed_ratio desc, limit top 20
result_sorted = result_df.sort_values(
    by=["total_anomalous_points", "max_exceed_ratio"], ascending=[False, False]
).reset_index(drop=True).head(20)

# Counts: number of groups scanned (rows in this table) and how many had number_of_anomalous_runs > 0
counts = pd.Series(
    {
        "num_groups_scanned": int(result_df.shape[0]),
        "groups_with_anomalous_runs": int((result_df["number_of_anomalous_runs"] > 0).sum()),
    }
)

# Display compact results
result_sorted[[
    "component",
    "kpi_name",
    "global_p95",
    "total_points_in_window",
    "total_anomalous_points",
    "number_of_anomalous_runs",
    "longest_run_length",
    "earliest_run_start_utc",
    "max_value_in_window",
    "max_exceed_ratio",
]], counts

```
Out[3]:
```
```python
# Plain-English summary string (displayed as the final result)
summary = (
    "Summary for the five candidate services (metric.csv, incident window 2024-01-15 21:21:06–21:51:06 UTC):\n\n"
    "Scanned groups: 34 (component,kpi) groups for the five services. "
    "26 groups had one or more anomalous runs (value > group-wide P95) during the window.\n\n"
    "Key findings (compact):\n"
    "- Highest relative spike: checkoutservice / latency-90 — max exceeded P95 by ~21% (max 1.2715 vs P95 1.0513). "
    "Earliest anomalous point: 2024-01-15 21:24:00 UTC. Longest anomalous run: 2 minutes.\n"
    "- Other notable spikes by exceed ratio: currencyservice / latency-90 (~9.6%, earliest 21:25), "
    "emailservice / workload (~7.9%, earliest 21:36), currencyservice / socket (~5.0%, earliest 21:25), "
    "checkoutservice / latency-50 (~4.4%, earliest 21:24).\n"
    "- Most anomalous groups show 25 total points in-window with 2 anomalous points each (i.e., brief spikes of 1–2 consecutive minutes). "
    "Some groups have 2 separate anomalous runs, others a single run; longest runs observed are typically 2 minutes.\n\n"
    "Interpretation:\n"
    "- The five candidate services experienced brief, correlated spikes across latency, workload, CPU, and socket KPIs rather than prolonged continuous degradation. "
    "Spikes are short (1–2 minutes) but visible across multiple KPIs and services, consistent with a transient workload/traffic surge causing short-lived resource/latency impacts.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                component    kpi_name    global_p95  total_points_in_window  total_anomalous_points  number_of_anomalous_runs  longest_run_length    earliest_run_start_utc  max_value_in_window  max_exceed_ratio
0         checkoutservice  latency-90  1.051278e+00                      25                       2                         1                   2 2024-01-15 21:24:00+00:00         1.271531e+00          0.209510
1         currencyservice  latency-90  1.184829e-01                      25                       2                         1                   2 2024-01-15 21:25:00+00:00         1.299076e-01          0.096425
2            emailservice    workload  1.229727e+00                      25                       2                         2                   1 2024-01-15 21:36:00+00:00         1.326683e+00          0.078844
3         currencyservice      socket  4.380000e+00                      25                       2                         2                   1 2024-01-15 21:25:00+00:00         4.600000e+00          0.050228
4         checkoutservice  latency-50  3.797247e-01                      25                       2                         1                   2 2024-01-15 21:24:00+00:00         3.962963e-01          0.043641
5         currencyservice    workload  7.868004e+01                      25                       2                         2                   1 2024-01-15 21:24:00+00:00         8.206359e+01          0.043004
6   productcatalogservice         cpu  3.593067e+00                      25                       2                         2                   1 2024-01-15 21:38:00+00:00         3.709160e+00          0.032310
7         checkoutservice    workload  1.252943e+00                      25                       2                         1                   2 2024-01-15 21:37:00+00:00         1.288700e+00          0.028538
8   productcatalogservice  latency-50  2.231006e-03                      25                       2                         1                   2 2024-01-15 21:32:00+00:00         2.283897e-03          0.023707
9   recommendationservice    workload  1.873964e+01                      25                       2                         2                   1 2024-01-15 21:32:00+00:00         1.905110e+01          0.016620
10        currencyservice  latency-50  6.450636e-02                      25                       2                         2                   1 2024-01-15 21:24:00+00:00         6.535791e-02          0.013201
11        currencyservice         cpu  1.881616e+01                      25                       2                         2                   1 2024-01-15 21:24:00+00:00         1.906427e+01          0.013186
12           emailservice         cpu  3.193352e-01                      25                       2                         1                   2 2024-01-15 21:37:00+00:00         3.229842e-01          0.011427
13        checkoutservice         cpu  1.978319e+01                      25                       2                         2                   1 2024-01-15 21:42:00+00:00         1.999501e+01          0.010707
14           emailservice  latency-90  4.713541e-03                      25                       2                         2                   1 2024-01-15 21:36:00+00:00         4.761981e-03          0.010277
15           emailservice  latency-50  3.063078e-03                      25                       2                         2                   1 2024-01-15 21:36:00+00:00         3.089989e-03          0.008786
16  productcatalogservice  latency-90  4.457786e-03                      25                       2                         1                   2 2024-01-15 21:32:00+00:00         4.494202e-03          0.008169
17  recommendationservice         cpu  2.622247e+00                      25                       2                         2                   1 2024-01-15 21:35:00+00:00         2.635759e+00          0.005153
18        currencyservice         mem  4.097597e+07                      25                       2                         2                   1 2024-01-15 21:37:00+00:00         4.111551e+07          0.003406
19  productcatalogservice    workload  1.435848e+02                      25                       2                         2                   1 2024-01-15 21:38:00+00:00         1.439564e+02          0.002588, num_groups_scanned            34
groups_with_anomalous_runs    26
dtype: int64)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse candidates list if available, otherwise define
try:
    candidates
except NameError:
    candidates = [
        "checkoutservice",
        "currencyservice",
        "emailservice",
        "productcatalogservice",
        "recommendationservice",
    ]

# 1) Read trace.csv and parse timestamp as UTC
df_trace = pd.read_csv("trace.csv")
df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# 2) Compute global P95 per (cmdb_id, trace_name) using the full series
p95_trace = (
    df_trace.groupby(["cmdb_id", "trace_name"])["value"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"value": "global_p95"})
)

# Restrict to candidate services for scanning/reporting
p95_candidates = p95_trace[p95_trace["cmdb_id"].isin(candidates)].reset_index(drop=True)

# 3) Filter trace.csv to the incident window (inclusive)
start = pd.to_datetime("2024-01-15 21:21:06", utc=True)
end = pd.to_datetime("2024-01-15 21:51:06", utc=True)
df_trace_window = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)].copy()

# 4) For rows where cmdb_id in candidates, mark anomaly points where value > global_P95
df_window_cand = df_trace_window[df_trace_window["cmdb_id"].isin(candidates)].copy()
# merge global_p95 into windowed candidate rows
df_window_cand = df_window_cand.merge(p95_candidates, on=["cmdb_id", "trace_name"], how="left")
df_window_cand["above_p95"] = df_window_cand["value"] > df_window_cand["global_p95"]

# 5) Aggregations per (cmdb_id, trace_name) among candidates
total_points = (
    df_window_cand.groupby(["cmdb_id", "trace_name"])["value"]
    .count()
    .rename("total_points_in_window")
    .reset_index()
)
points_above = (
    df_window_cand.groupby(["cmdb_id", "trace_name"])["above_p95"]
    .sum()
    .rename("points_above_p95")
    .reset_index()
)
earliest_anom = (
    df_window_cand[df_window_cand["above_p95"]]
    .groupby(["cmdb_id", "trace_name"])["timestamp"]
    .min()
    .rename("earliest_anomaly_timestamp_utc")
    .reset_index()
)
max_value = (
    df_window_cand.groupby(["cmdb_id", "trace_name"])["value"]
    .max()
    .rename("max_value_in_window")
    .reset_index()
)

# Start from p95_candidates (all groups for the five services) and left-join window aggregates
merged = (
    p95_candidates.merge(total_points, on=["cmdb_id", "trace_name"], how="left")
    .merge(points_above, on=["cmdb_id", "trace_name"], how="left")
    .merge(earliest_anom, on=["cmdb_id", "trace_name"], how="left")
    .merge(max_value, on=["cmdb_id", "trace_name"], how="left")
)

# Fill NaNs for counts with 0 and keep timestamps as NaT and max_value as NaN if missing
merged["total_points_in_window"] = merged["total_points_in_window"].fillna(0).astype(int)
merged["points_above_p95"] = merged["points_above_p95"].fillna(0).astype(int)

# Rename for output formatting
merged = merged.rename(columns={"cmdb_id": "component", "trace_name": "trace_name"})

# Select output columns
out = merged[
    [
        "component",
        "trace_name",
        "global_p95",
        "total_points_in_window",
        "points_above_p95",
        "earliest_anomaly_timestamp_utc",
        "max_value_in_window",
    ]
]

# 5) Sort and limit to top 20 rows
trace_summary_top = out.sort_values(
    by=["points_above_p95", "max_value_in_window"], ascending=[False, False]
).head(20).reset_index(drop=True)

# 6) Counts: number of groups scanned among these five services and how many had points_above_p95 > 0
trace_counts = pd.Series(
    {
        "num_groups_scanned": int(p95_candidates.shape[0]),
        "groups_with_points_above_p95": int((merged["points_above_p95"] > 0).sum()),
    }
)

# Return compact results (DataFrame and counts)
trace_summary_top, trace_counts

```
Out[4]:
```
Summary for the five candidate services using trace.csv (incident window 2024-01-15 21:21:06–21:51:06 UTC):

- Scan counts: 88 (cmdb_id,trace_name) groups for the five services were scanned; 62 groups had points above their group-wide P95 during the window.

- Top signals (compact):
  - productcatalogservice — trace.from_frontendservice.row_count: global P95 3675, 25 points in-window, 2 points above P95, earliest anomaly 2024-01-15 21:25:00, max 3699.
  - currencyservice — trace.from_frontendservice.row_count: global P95 2296.4, 25 points, 2 above P95, earliest 21:33, max 2337.
  - recommendationservice & productcatalogservice — multiple row_count traces (from/to recommendationservice, to_productcatalogservice, etc.) show the same pattern: 25 points, 2 above P95, earliest ~21:25, small increases above P95 (e.g., max 564 vs P95 562).
  - checkoutservice — several row_count traces and duration metrics show anomalies (e.g., trace.from_checkoutservice.row_count P95 304.4, earliest anomaly 21:36, max 353; also duration_p95 and duration_mean show small increases starting ~21:24).

- Overall interpretation:
  - The dominant trace signal is elevated row_count (request counts) across frontend → downstream services (productcatalogservice, recommendationservice, currencyservice, checkoutservice, emailservice). Most affected traces show 25 samples in-window with 1–2 brief anomalous points above the group P95.
  - This pattern is consistent with a short-lived surge in traffic/requests impacting multiple services (increased request counts and modest increases in trace durations), rather than isolated service failures.

The original code execution output of IPython Kernel is also provided below for reference:

(                component                                  trace_name   global_p95  total_points_in_window  points_above_p95 earliest_anomaly_timestamp_utc  max_value_in_window
0   productcatalogservice        trace.from_frontendservice.row_count  3675.000000                      25                 2      2024-01-15 21:25:00+00:00          3699.000000
1         currencyservice        trace.from_frontendservice.row_count  2296.400000                      25                 2      2024-01-15 21:33:00+00:00          2337.000000
2   productcatalogservice  trace.from_recommendationservice.row_count   562.000000                      25                 2      2024-01-15 21:25:00+00:00           564.000000
3   recommendationservice        trace.from_frontendservice.row_count   562.000000                      25                 2      2024-01-15 21:25:00+00:00           564.000000
4   recommendationservice  trace.from_recommendationservice.row_count   562.000000                      25                 2      2024-01-15 21:25:00+00:00           564.000000
5   recommendationservice    trace.to_productcatalogservice.row_count   562.000000                      25                 2      2024-01-15 21:25:00+00:00           564.000000
6   recommendationservice    trace.to_recommendationservice.row_count   562.000000                      25                 2      2024-01-15 21:25:00+00:00           564.000000
7         checkoutservice        trace.from_checkoutservice.row_count   304.400000                      25                 2      2024-01-15 21:36:00+00:00           353.000000
8         checkoutservice          trace.to_checkoutservice.row_count   304.400000                      25                 2      2024-01-15 21:36:00+00:00           353.000000
9         checkoutservice                   trace.from_root.row_count   147.000000                      25                 2      2024-01-15 21:30:00+00:00           168.000000
10        checkoutservice          trace.to_currencyservice.row_count   122.200000                      25                 2      2024-01-15 21:30:00+00:00           145.000000
11        currencyservice        trace.from_checkoutservice.row_count   122.200000                      25                 2      2024-01-15 21:30:00+00:00           145.000000
12        checkoutservice    trace.to_productcatalogservice.row_count    90.200000                      25                 2      2024-01-15 21:30:00+00:00           103.000000
13  productcatalogservice        trace.from_checkoutservice.row_count    90.200000                      25                 2      2024-01-15 21:30:00+00:00           103.000000
14        checkoutservice             trace.to_emailservice.row_count    37.400000                      25                 2      2024-01-15 21:36:00+00:00            42.000000
15        checkoutservice           trace.to_paymentservice.row_count    37.400000                      25                 2      2024-01-15 21:36:00+00:00            42.000000
16           emailservice        trace.from_checkoutservice.row_count    37.400000                      25                 2      2024-01-15 21:36:00+00:00            42.000000
17        checkoutservice        trace.from_frontendservice.row_count    37.400000                      25                 2      2024-01-15 21:36:00+00:00            41.000000
18        checkoutservice     trace.from_frontendservice.duration_p95     0.944224                      25                 2      2024-01-15 21:24:00+00:00             1.172375
19        checkoutservice    trace.from_frontendservice.duration_mean     0.462834                      25                 2      2024-01-15 21:24:00+00:00             0.527776, num_groups_scanned              88
groups_with_points_above_p95    62
dtype: int64)```
```